# SemCons-IDS — Best Paper Final Edition v7

Version finale intégrant les corrections méthodologiques, les analyses de généralisabilité, la conformité, l'explicabilité, les visualisations article-ready et les tableaux d'efficacité.

## 0 — Installation compatible Google Colab

In [ ]:
import os
import sys
import signal
import subprocess
import importlib.metadata as metadata

PINNED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}

def version_of(name: str):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def stack_ok() -> bool:
    code = (
        "from transformers import AutoTokenizer, AutoModelForSequenceClassification;"
        "from peft import LoraConfig, TaskType, get_peft_model;"
        "from xgboost import XGBClassifier;"
        "print('STACK_OK')"
    )
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True,
        capture_output=True,
    )
    return result.returncode == 0 and "STACK_OK" in result.stdout

installed = {name: version_of(name) for name in PINNED}
torchao_version = version_of("torchao")

needs_repair = (
    torchao_version is not None
    or any(installed[name] != expected for name, expected in PINNED.items())
    or not stack_ok()
)

print("Versions avant vérification :")
for name, expected in PINNED.items():
    print(f"  {name:14s}: {installed[name]} (attendu : {expected})")
print(f"  {'torchao':14s}: {torchao_version or 'absent'}")

if needs_repair:
    print("\nRéinstallation propre de la pile...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "transformers", "peft", "accelerate", "tokenizers", "torchao", "xgboost",
        ],
        check=False,
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--no-cache-dir",
            "transformers==4.57.6",
            "peft==0.18.0",
            "accelerate==1.12.0",
            "xgboost==2.1.4",
            "pyarrow>=15.0.0",
            "imbalanced-learn>=0.12.0",
            "joblib>=1.3.0",
            "psutil>=5.9.0",
            "matplotlib>=3.8.0",
            "seaborn>=0.13.2",
        ]
    )
    if not stack_ok():
        raise RuntimeError("La pile réinstallée n'est pas importable.")
    print("✅ Installation réparée.")
    print("♻️ Redémarrage automatique du noyau...")
    os.kill(os.getpid(), signal.SIGKILL)

print("✅ Environnement stable et prêt.")

## 1 — Imports, configuration et dossiers de sortie

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import importlib.metadata as metadata
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from xgboost import XGBClassifier

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None

warnings.filterwarnings("ignore")
sns.set_context("talk")

EXPECTED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}
for package, expected in EXPECTED.items():
    found = metadata.version(package)
    if found != expected:
        raise RuntimeError(
            f"{package}={found}; attendu={expected}. "
            "Réexécutez la cellule d'installation et laissez le noyau redémarrer."
        )

@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Dataset/CIC-IDS-2017/"
    output_dir: str = "/content/drive/MyDrive/these_llm/SemCons_IDS_BestPaper_Final_v7/"
    model_checkpoint: str = "sentence-transformers/all-MiniLM-L6-v2"

    seeds: List[int] = field(default_factory=lambda: [42, 123, 2026])
    protocols: List[str] = field(default_factory=lambda: ["iid_stratified", "temporal_within_file"])

    train_fraction: float = 0.70
    val_fraction: float = 0.10
    cal_fraction: float = 0.10
    test_fraction: float = 0.10

    min_samples_per_class: int = 200
    numeric_top_k: int = 60
    text_top_k: int = 16
    feature_selection_rows_per_class: int = 20_000

    llm_train_cap_per_class: int = 20_000
    llm_val_cap_per_class: int = 5_000
    max_length: int = 320

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05

    epochs: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    batch_train: int = 32
    batch_eval: int = 128
    grad_accum: int = 1
    warmup_fraction: float = 0.06
    label_smoothing: float = 0.01
    early_stopping_patience: int = 2

    use_sqrt_class_weights: bool = True
    class_weight_min: float = 0.60
    class_weight_max: float = 2.50

    xgb_n_estimators: int = 1200
    xgb_max_depth: int = 12
    xgb_learning_rate: float = 0.08
    xgb_subsample: float = 0.90
    xgb_colsample_bytree: float = 0.90
    xgb_min_child_weight: float = 1.0
    xgb_reg_lambda: float = 1.0
    xgb_early_stopping_rounds: int = 50

    fusion_selection_metric: str = "weighted_f1"
    fusion_min_semantic_weight: float = 0.00
    fusion_grid_points: int = 41

    target_weighted_f1: float = 0.99
    selection_gain_margin: float = 5e-4

    conformal_alpha: float = 0.10
    raps_lambda: float = 0.02
    raps_k_reg: int = 2
    conformal_abstain_size: int = 2

    explanation_model_checkpoint: str = "google/flan-t5-small"
    explanation_samples: int = 60
    slm_explanation_samples: int = 30
    explanation_top_k: int = 5
    explanation_max_new_tokens: int = 96
    explanation_language: str = "English"

    figure_dpi: int = 220

CFG = Config()

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG.seeds[0])

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive non monté :", exc)

OUTPUT_DIR = Path(CFG.output_dir)
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")

plt.rcParams["figure.dpi"] = CFG.figure_dpi
plt.rcParams["savefig.dpi"] = CFG.figure_dpi

def save_figure(filename_stem: str) -> None:
    png = FIG_DIR / f"{filename_stem}.png"
    pdf = FIG_DIR / f"{filename_stem}.pdf"
    plt.tight_layout()
    plt.savefig(png, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Figure sauvegardée :", png)

## 2 — Chargement, normalisation des labels et déduplication

In [ ]:
def normalize_column_name(name: str) -> str:
    value = str(name).strip().lower()
    value = re.sub(r"[\s/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

def normalize_label(value: object) -> str | None:
    raw = str(value).strip()
    key = re.sub(r"\s+", " ", raw.lower().replace("_", " ")).strip()
    mapping = {
        "benign": "BENIGN",
        "bot": "Bot",
        "ddos": "DDoS",
        "dos hulk": "DoS_Hulk",
        "dos goldeneye": "DoS_GoldenEye",
        "dos slowhttptest": "DoS_Slowhttptest",
        "dos slowloris": "DoS_Slowloris",
        "ftp-patator": "FTP-Patator",
        "ftp patator": "FTP-Patator",
        "ssh-patator": "SSH-Patator",
        "ssh patator": "SSH-Patator",
        "portscan": "PortScan",
        "port scan": "PortScan",
        "infiltration": "Infiltration",
        "heartbleed": "Heartbleed",
    }
    return mapping.get(key)

def find_data_files(data_dir: str) -> List[Path]:
    root = Path(data_dir)
    files = []
    for pattern in ("*.parquet", "*.csv", "*.feather"):
        files.extend(root.glob(pattern))
    return sorted(files)

def read_one_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix == ".feather":
        frame = pd.read_feather(path)
    elif suffix == ".csv":
        frame = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(path)
    frame.columns = [normalize_column_name(c) for c in frame.columns]
    frame["__source_file__"] = path.name
    frame["__row_in_file__"] = np.arange(len(frame))
    return frame

files = find_data_files(CFG.data_dir)
if not files:
    raise FileNotFoundError(f"Aucun CSV/Parquet/Feather dans {CFG.data_dir}")

frames = []
for file_path in files:
    frame = read_one_file(file_path)
    frames.append(frame)
    print(f"{file_path.name:<70s} {len(frame):>10,} lignes")

df = pd.concat(frames, ignore_index=True)

label_candidates = [c for c in df.columns if c == "label" or "label" in c or "class" in c]
if not label_candidates:
    raise ValueError("Colonne label introuvable.")
LABEL_COL = label_candidates[0]

df[LABEL_COL] = df[LABEL_COL].map(normalize_label)
unmapped = int(df[LABEL_COL].isna().sum())
df = df.dropna(subset=[LABEL_COL]).copy()

before_dedup = len(df)
dedup_columns = [c for c in df.columns if c not in {"__source_file__", "__row_in_file__"}]
df = df.drop_duplicates(subset=dedup_columns).reset_index(drop=True)

feature_candidates = [
    c for c in df.columns
    if c not in {LABEL_COL, "__source_file__", "__row_in_file__"}
]
for column in feature_candidates:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[feature_candidates] = df[feature_candidates].replace([np.inf, -np.inf], np.nan)

counts = df[LABEL_COL].value_counts()
keep_classes = counts[counts >= CFG.min_samples_per_class].index.tolist()
removed_classes = sorted(set(counts.index) - set(keep_classes))
df = df[df[LABEL_COL].isin(keep_classes)].reset_index(drop=True)

print(f"\nLabels non reconnus supprimés : {unmapped:,}")
print(f"Doublons supprimés            : {before_dedup - len(df):,}")
print(f"Dataset retenu                : {len(df):,} lignes × {len(feature_candidates)} variables candidates")
print("Classes supprimées            :", removed_classes)
display(df[LABEL_COL].value_counts().to_frame("effectif"))

ALL_CLASS_COUNTS = df[LABEL_COL].value_counts().rename_axis("label").reset_index(name="count")
ALL_CLASS_COUNTS.to_csv(TABLE_DIR / "overall_class_distribution.csv", index=False)

## 3 — Graphe global : distribution des classes

In [ ]:
distribution_plot_df = ALL_CLASS_COUNTS.sort_values("count", ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(distribution_plot_df["label"], distribution_plot_df["count"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Nombre de flux")
plt.title("Distribution globale des classes — CICIDS2017 nettoyé")
save_figure("global_class_distribution")
plt.show()

## 4 — Fonctions de split, prétraitement, MiniLM et XGBoost

In [ ]:
def make_splits(
    data: pd.DataFrame,
    protocol: str,
    seed: int,
    label_col: str,
    train_fraction: float,
    val_fraction: float,
    cal_fraction: float,
    test_fraction: float,
) -> Dict[str, pd.DataFrame]:
    if not math.isclose(train_fraction + val_fraction + cal_fraction + test_fraction, 1.0, rel_tol=0, abs_tol=1e-9):
        raise ValueError("Les fractions de split doivent sommer à 1.0.")

    if protocol == "iid_stratified":
        train_df, temp_df = train_test_split(
            data,
            test_size=1.0 - train_fraction,
            stratify=data[label_col],
            random_state=seed,
        )
        relative_val = val_fraction / (val_fraction + cal_fraction + test_fraction)
        val_df, remain_df = train_test_split(
            temp_df,
            test_size=1.0 - relative_val,
            stratify=temp_df[label_col],
            random_state=seed + 1,
        )
        relative_cal = cal_fraction / (cal_fraction + test_fraction)
        cal_df, test_df = train_test_split(
            remain_df,
            test_size=1.0 - relative_cal,
            stratify=remain_df[label_col],
            random_state=seed + 2,
        )
        splits = {
            "train": train_df.reset_index(drop=True),
            "val": val_df.reset_index(drop=True),
            "cal": cal_df.reset_index(drop=True),
            "test": test_df.reset_index(drop=True),
        }
        return splits

    if protocol == "temporal_within_file":
        buckets = {"train": [], "val": [], "cal": [], "test": []}
        for source_file, group in data.groupby("__source_file__", sort=False):
            group = group.sort_values("__row_in_file__").reset_index(drop=True)
            n = len(group)

            train_end = int(round(train_fraction * n))
            val_end = train_end + int(round(val_fraction * n))
            cal_end = val_end + int(round(cal_fraction * n))

            # Garantir au moins une ligne quand c'est possible.
            train_end = max(1, min(train_end, n))
            val_end = max(train_end + 1, min(val_end, n)) if n >= 2 else train_end
            cal_end = max(val_end + 1, min(cal_end, n)) if n >= 3 else val_end

            buckets["train"].append(group.iloc[:train_end])
            buckets["val"].append(group.iloc[train_end:val_end])
            buckets["cal"].append(group.iloc[val_end:cal_end])
            buckets["test"].append(group.iloc[cal_end:])

        splits = {
            name: pd.concat(parts, ignore_index=True).reset_index(drop=True)
            for name, parts in buckets.items()
        }
        return splits

    raise ValueError(f"Protocole inconnu : {protocol}")

def capped_indices_per_class(y: np.ndarray, cap: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected = []
    for cls in np.unique(y):
        indices = np.flatnonzero(y == cls)
        if len(indices) > cap:
            indices = rng.choice(indices, size=cap, replace=False)
        selected.append(indices)
    return np.sort(np.concatenate(selected))

def prepare_split_artifacts(
    splits: Dict[str, pd.DataFrame],
    cfg: Config,
    label_col: str,
) -> Dict[str, object]:
    label_encoder = LabelEncoder()
    label_encoder.fit(splits["train"][label_col])
    class_names = label_encoder.classes_.tolist()
    n_classes = len(class_names)

    all_feature_columns = [
        c for c in splits["train"].columns
        if c not in {label_col, "__source_file__", "__row_in_file__"}
    ]

    X_train_all = splits["train"][all_feature_columns].to_numpy(dtype=np.float64)
    y_train_all = label_encoder.transform(splits["train"][label_col])

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_all)

    variance_filter = VarianceThreshold(0.0)
    X_train_var = variance_filter.fit_transform(X_train_imp)

    variance_features = [
        feature
        for feature, keep in zip(all_feature_columns, variance_filter.get_support())
        if keep
    ]

    selection_indices = capped_indices_per_class(
        y_train_all,
        cfg.feature_selection_rows_per_class,
        cfg.seeds[0],
    )

    selector = RandomForestClassifier(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=cfg.seeds[0],
    )
    selector.fit(X_train_var[selection_indices], y_train_all[selection_indices])

    importance_df = pd.DataFrame({
        "feature": variance_features,
        "importance": selector.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    numeric_features = importance_df.head(min(cfg.numeric_top_k, len(importance_df)))["feature"].tolist()

    # Éviter quelques redondances ou doublons directs.
    cleaned_text_features = []
    seen_prefixes = set()
    for feature in numeric_features:
        prefix = feature.replace("1", "")
        if prefix in seen_prefixes:
            continue
        seen_prefixes.add(prefix)
        cleaned_text_features.append(feature)
        if len(cleaned_text_features) >= cfg.text_top_k:
            break
    text_features = cleaned_text_features

    variance_index = {name: idx for idx, name in enumerate(variance_features)}
    numeric_positions = [variance_index[name] for name in numeric_features]
    text_positions = [numeric_features.index(name) for name in text_features]

    scaler = StandardScaler()

    arrays = {}
    for split_name, split_df in splits.items():
        X = split_df[all_feature_columns].to_numpy(dtype=np.float64)
        X_imp = imputer.transform(X)
        X_var = variance_filter.transform(X_imp)
        X_num = X_var[:, numeric_positions].astype(np.float32)
        if split_name == "train":
            X_scaled = scaler.fit_transform(X_num).astype(np.float32)
        else:
            X_scaled = scaler.transform(X_num).astype(np.float32)
        arrays[split_name] = {
            "raw_numeric": X_num,
            "scaled_numeric": X_scaled,
            "scaled_text": X_scaled[:, text_positions],
            "y": label_encoder.transform(split_df[label_col]).astype(np.int64),
        }

    return {
        "label_encoder": label_encoder,
        "class_names": class_names,
        "n_classes": n_classes,
        "all_feature_columns": all_feature_columns,
        "numeric_features": numeric_features,
        "text_features": text_features,
        "importance_df": importance_df,
        "imputer": imputer,
        "variance_filter": variance_filter,
        "scaler": scaler,
        "arrays": arrays,
    }

def compact_prompt(text_features: List[str], values: np.ndarray) -> str:
    return "classify: " + " ".join(
        f"{name}={float(value):.4f}"
        for name, value in zip(text_features, values)
    )

class IndexedDataset(Dataset):
    def __init__(self, labels: np.ndarray, indices: np.ndarray):
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item: int):
        idx = int(self.indices[item])
        return {"index": idx, "label": int(self.labels[idx])}

class CompactCollator:
    def __init__(self, tokenizer, scaled_text: np.ndarray, text_features: List[str], max_length: int):
        self.tokenizer = tokenizer
        self.scaled_text = scaled_text
        self.text_features = text_features
        self.max_length = max_length

    def __call__(self, items):
        indices = [item["index"] for item in items]
        labels = torch.tensor([item["label"] for item in items], dtype=torch.long)
        texts = [compact_prompt(self.text_features, self.scaled_text[idx]) for idx in indices]
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        encoded["labels"] = labels
        encoded["indices"] = torch.tensor(indices, dtype=torch.long)
        return encoded



def metrics_from_probs(probabilities: np.ndarray, labels: np.ndarray, n_classes: int) -> Dict[str, float]:
    predictions = probabilities.argmax(axis=1)
    all_labels = np.arange(n_classes)
    present_labels = np.unique(labels)

    macro_f1_present = float(f1_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_f1_all = float(f1_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))
    macro_precision_present = float(precision_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_precision_all = float(precision_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))
    macro_recall_present = float(recall_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_recall_all = float(recall_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": macro_f1_present,
        "macro_f1_present": macro_f1_present,
        "macro_f1_all": macro_f1_all,
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "macro_precision": macro_precision_present,
        "macro_precision_present": macro_precision_present,
        "macro_precision_all": macro_precision_all,
        "macro_recall": macro_recall_present,
        "macro_recall_present": macro_recall_present,
        "macro_recall_all": macro_recall_all,
        "nll": float(log_loss(labels, probabilities, labels=all_labels)),
        "present_class_count": int(len(present_labels)),
        "label_space_coverage": float(len(present_labels) / max(1, n_classes)),
    }


def dir_size_mb(path: Path) -> float:
    return float(sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / (1024 ** 2))


def build_class_weights(y_train_small: np.ndarray, n_classes: int, cfg: Config) -> torch.Tensor:
    counts = np.bincount(y_train_small, minlength=n_classes).astype(np.float64)
    if cfg.use_sqrt_class_weights:
        weights = np.sqrt(
            len(y_train_small) / (n_classes * np.maximum(counts, 1.0))
        )
        weights = np.clip(weights, cfg.class_weight_min, cfg.class_weight_max)
        weights = weights / weights.mean()
    else:
        weights = np.ones(n_classes, dtype=np.float64)
    return torch.tensor(weights, dtype=torch.float32)

def build_minilm_model(model_checkpoint: str, n_classes: int, class_names: List[str], cfg: Config):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_classes,
        id2label={i: name for i, name in enumerate(class_names)},
        label2id={name: i for i, name in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )
    linear_leaf_names = sorted({
        name.split(".")[-1]
        for name, module in base_model.named_modules()
        if isinstance(module, nn.Linear)
    })
    target_modules = ["query", "value"]
    missing = [name for name in target_modules if name not in linear_leaf_names]
    if missing:
        raise ValueError(f"Modules LoRA introuvables : {missing}")

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=target_modules,
        modules_to_save=["classifier"],
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(base_model, lora_config).to(DEVICE)

    n_total = sum(parameter.numel() for parameter in model.parameters())
    n_trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    n_lora = sum(
        parameter.numel()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad and "lora_" in name
    )
    n_head = n_trainable - n_lora

    stats = {
        "llm_total_parameters": n_total,
        "llm_lora_parameters": n_lora,
        "llm_head_parameters": n_head,
        "llm_trainable_parameters": n_trainable,
        "llm_trainable_percent": 100.0 * n_trainable / n_total,
    }
    return model, stats

@torch.no_grad()
def collect_llm_probs(
    model,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    for batch in tqdm(loader, leave=False):
        labels = batch.pop("labels")
        indices = batch.pop("indices")
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
        all_indices.append(indices.numpy())
    return (
        np.concatenate(all_probs),
        np.concatenate(all_labels),
        np.concatenate(all_indices),
    )

def train_minilm(
    arrays: Dict[str, Dict[str, np.ndarray]],
    text_features: List[str],
    class_names: List[str],
    n_classes: int,
    seed: int,
    run_dir: Path,
    cfg: Config,
) -> Dict[str, object]:
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_checkpoint)

    train_indices = capped_indices_per_class(arrays["train"]["y"], cfg.llm_train_cap_per_class, seed + 10)
    val_indices = capped_indices_per_class(arrays["val"]["y"], cfg.llm_val_cap_per_class, seed + 11)

    class_weights = build_class_weights(arrays["train"]["y"][train_indices], n_classes, cfg)

    train_dataset = IndexedDataset(arrays["train"]["y"], train_indices)
    val_dataset = IndexedDataset(arrays["val"]["y"], val_indices)

    train_collator = CompactCollator(tokenizer, arrays["train"]["scaled_text"], text_features, cfg.max_length)
    val_collator = CompactCollator(tokenizer, arrays["val"]["scaled_text"], text_features, cfg.max_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_train,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=train_collator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=val_collator,
    )

    model, param_stats = build_minilm_model(cfg.model_checkpoint, n_classes, class_names, cfg)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=cfg.label_smoothing,
    )
    optimizer = AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum)
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_fraction * total_steps)

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
    amp_enabled = DEVICE.type == "cuda"
    grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    best_state = None
    best_epoch = -1
    best_macro = -np.inf
    patience = 0
    history = []

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    train_start = time.perf_counter()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(train_loader, desc=f"MiniLM seed={seed} epoch {epoch}/{cfg.epochs}")
        for step, batch in enumerate(progress, start=1):
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch.pop("indices")
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(**batch).logits
                loss = criterion(logits, labels) / cfg.grad_accum

            grad_scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * cfg.grad_accum

            if step % cfg.grad_accum == 0 or step == len(train_loader):
                grad_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [parameter for parameter in model.parameters() if parameter.requires_grad],
                    max_norm=1.0,
                )
                grad_scaler.step(optimizer)
                grad_scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            progress.set_postfix(loss=f"{running_loss/step:.4f}")

        val_probs_small, val_labels_small, _ = collect_llm_probs(model, val_loader, DEVICE)
        val_metrics_small = metrics_from_probs(val_probs_small, val_labels_small, n_classes)
        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(1, len(train_loader)),
            **{f"val_bal_{key}": value for key, value in val_metrics_small.items()},
        }
        history.append(row)
        print(pd.Series(row).round(6))

        if val_metrics_small["macro_f1"] > best_macro:
            best_macro = val_metrics_small["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience = 0
            print(f"🏆 Nouveau meilleur MiniLM (seed={seed}) : Macro-F1={best_macro:.6f}")
        else:
            patience += 1
            if patience >= cfg.early_stopping_patience:
                print("Early stopping MiniLM.")
                break

    if best_state is None:
        raise RuntimeError("Aucun checkpoint MiniLM sauvegardé.")

    model.load_state_dict(best_state, strict=True)

    llm_train_seconds = time.perf_counter() - train_start
    peak_vram_gib = (
        torch.cuda.max_memory_allocated() / 2**30
        if DEVICE.type == "cuda"
        else 0.0
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "minilm_training_history.csv", index=False)

    return {
        "tokenizer": tokenizer,
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_balanced_val_macro_f1": best_macro,
        "train_seconds": llm_train_seconds,
        "peak_vram_gib": peak_vram_gib,
        "param_stats": param_stats,
        "train_indices": train_indices,
        "val_indices": val_indices,
        "class_weights": class_weights.numpy(),
    }

def train_xgboost(
    arrays: Dict[str, Dict[str, np.ndarray]],
    n_classes: int,
    seed: int,
    cfg: Config,
) -> Tuple[object, float]:
    xgb_common = dict(
        objective="multi:softprob",
        num_class=n_classes,
        n_estimators=cfg.xgb_n_estimators,
        max_depth=cfg.xgb_max_depth,
        learning_rate=cfg.xgb_learning_rate,
        subsample=cfg.xgb_subsample,
        colsample_bytree=cfg.xgb_colsample_bytree,
        min_child_weight=cfg.xgb_min_child_weight,
        reg_lambda=cfg.xgb_reg_lambda,
        max_bin=256,
        tree_method="hist",
        eval_metric="mlogloss",
        early_stopping_rounds=cfg.xgb_early_stopping_rounds,
        random_state=seed,
        n_jobs=-1,
    )

    model = XGBClassifier(
        **xgb_common,
        device="cuda" if DEVICE.type == "cuda" else "cpu",
    )

    start = time.perf_counter()
    try:
        model.fit(
            arrays["train"]["raw_numeric"],
            arrays["train"]["y"],
            eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
            verbose=False,
        )
    except Exception as exc:
        print("⚠️ XGBoost GPU indisponible, reprise CPU :", exc)
        model = XGBClassifier(**xgb_common, device="cpu")
        model.fit(
            arrays["train"]["raw_numeric"],
            arrays["train"]["y"],
            eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
            verbose=False,
        )
    seconds = time.perf_counter() - start
    return model, seconds

def selection_score(metrics: Dict[str, float], cfg: Config) -> float:
    if cfg.fusion_selection_metric not in metrics:
        raise KeyError(cfg.fusion_selection_metric)
    return metrics[cfg.fusion_selection_metric]

def make_full_llm_loader(
    arrays: Dict[str, Dict[str, np.ndarray]],
    split_name: str,
    tokenizer,
    text_features: List[str],
    cfg: Config,
) -> DataLoader:
    indices = np.arange(len(arrays[split_name]["y"]))
    dataset = IndexedDataset(arrays[split_name]["y"], indices)
    collator = CompactCollator(tokenizer, arrays[split_name]["scaled_text"], text_features, cfg.max_length)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
    )

def align_probabilities(
    probabilities: np.ndarray,
    source_classes: np.ndarray,
    n_classes: int,
) -> np.ndarray:
    """Aligne les colonnes d'un classifieur entraîné sur un sous-ensemble de classes."""
    probabilities = np.asarray(probabilities, dtype=np.float64)
    source_classes = np.asarray(source_classes, dtype=np.int64)
    aligned = np.zeros((len(probabilities), n_classes), dtype=np.float64)
    aligned[:, source_classes] = probabilities
    row_sums = aligned.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0
    if np.any(zero_rows):
        aligned[zero_rows] = 1.0 / n_classes
        row_sums = aligned.sum(axis=1, keepdims=True)
    return aligned / row_sums


def split_selection_and_conformal_indices(labels: np.ndarray, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    indices = np.arange(len(labels))
    unique, counts = np.unique(labels, return_counts=True)
    stratify = labels if len(unique) > 1 and counts.min() >= 2 else None
    select_idx, conformal_idx = train_test_split(
        indices,
        test_size=0.50,
        random_state=seed + 31,
        stratify=stratify,
    )
    return np.sort(select_idx), np.sort(conformal_idx)


## 5 — Fonction principale d’expérience

In [ ]:
def run_single_experiment(
    data: pd.DataFrame,
    protocol: str,
    seed: int,
    cfg: Config,
    output_dir: Path,
) -> Dict[str, object]:
    set_seed(seed)

    run_name = f"{protocol}__seed_{seed}"
    run_dir = output_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print(f"RUN : {run_name}")
    print("=" * 90)

    splits = make_splits(
        data=data,
        protocol=protocol,
        seed=seed,
        label_col=LABEL_COL,
        train_fraction=cfg.train_fraction,
        val_fraction=cfg.val_fraction,
        cal_fraction=cfg.cal_fraction,
        test_fraction=cfg.test_fraction,
    )

    split_sizes = []
    for split_name, split_df in splits.items():
        counts = split_df[LABEL_COL].value_counts().rename_axis("label").reset_index(name="count")
        counts["split"] = split_name
        counts["protocol"] = protocol
        counts["seed"] = seed
        split_sizes.append(counts)
        print(f"\n{split_name.upper():5s}: {len(split_df):,} lignes")
        print(split_df[LABEL_COL].value_counts().to_string())
    split_distribution_df = pd.concat(split_sizes, ignore_index=True)
    split_distribution_df.to_csv(run_dir / "split_distribution.csv", index=False)

    artifacts = prepare_split_artifacts(splits, cfg, LABEL_COL)
    arrays = artifacts["arrays"]
    class_names = artifacts["class_names"]
    n_classes = artifacts["n_classes"]
    numeric_features = artifacts["numeric_features"]
    text_features = artifacts["text_features"]
    importance_df = artifacts["importance_df"]

    importance_df.to_csv(run_dir / "feature_selection_importance.csv", index=False)

    # Audit de longueur des tokens sur un sous-échantillon train.
    audit_size = min(10_000, len(arrays["train"]["y"]))
    audit_indices = np.arange(audit_size)
    tokenizer_audit = AutoTokenizer.from_pretrained(cfg.model_checkpoint)
    token_lengths = []
    for idx in tqdm(audit_indices, desc=f"Audit tokens {run_name}"):
        prompt = compact_prompt(text_features, arrays["train"]["scaled_text"][idx])
        token_lengths.append(len(tokenizer_audit(prompt, truncation=False, add_special_tokens=True)["input_ids"]))
    token_lengths = np.asarray(token_lengths, dtype=np.int64)
    token_audit = {
        "protocol": protocol,
        "seed": seed,
        "n_audit": int(audit_size),
        "text_feature_count": int(len(text_features)),
        "max_length": int(cfg.max_length),
        "mean_tokens": float(token_lengths.mean()),
        "median_tokens": float(np.median(token_lengths)),
        "p95_tokens": float(np.percentile(token_lengths, 95)),
        "max_tokens": int(token_lengths.max()),
        "truncation_rate": float(np.mean(token_lengths > cfg.max_length)),
    }
    pd.DataFrame([token_audit]).to_csv(run_dir / "token_audit.csv", index=False)

    # MiniLM
    llm_bundle = train_minilm(arrays, text_features, class_names, n_classes, seed, run_dir, cfg)
    llm_model = llm_bundle["model"]
    llm_tokenizer = llm_bundle["tokenizer"]

    # XGBoost
    xgb_model, xgb_train_seconds = train_xgboost(arrays, n_classes, seed, cfg)

    probabilities = {}
    for split_name in ["val", "cal", "test"]:
        llm_loader = make_full_llm_loader(arrays, split_name, llm_tokenizer, text_features, cfg)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        llm_probs, labels, indices = collect_llm_probs(llm_model, llm_loader, DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        llm_seconds = time.perf_counter() - start

        xgb_probs = xgb_model.predict_proba(arrays[split_name]["raw_numeric"])
        xgb_probs = np.asarray(xgb_probs, dtype=np.float64)
        xgb_probs = xgb_probs / xgb_probs.sum(axis=1, keepdims=True)

        probabilities[split_name] = {
            "llm": llm_probs.astype(np.float64),
            "xgb": xgb_probs,
            "labels": labels,
            "indices": indices,
            "llm_seconds": llm_seconds,
        }

    # CAL est séparé : une moitié pour sélectionner la fusion, l'autre pour la conformité.
    cal_labels_all = probabilities["cal"]["labels"]
    cal_llm_all = probabilities["cal"]["llm"]
    cal_xgb_all = probabilities["cal"]["xgb"]
    cal_select_idx, cal_conformal_idx = split_selection_and_conformal_indices(cal_labels_all, seed)

    cal_labels = cal_labels_all[cal_select_idx]
    cal_llm = cal_llm_all[cal_select_idx]
    cal_xgb = cal_xgb_all[cal_select_idx]

    np.savez(
        run_dir / "calibration_partition_indices.npz",
        selection_indices=cal_select_idx,
        conformal_indices=cal_conformal_idx,
    )

    candidate_cal_metrics: Dict[str, Dict[str, float]] = {}
    candidate_test_probs: Dict[str, np.ndarray] = {}

    candidate_cal_metrics["minilm"] = metrics_from_probs(cal_llm, cal_labels, n_classes)
    candidate_test_probs["minilm"] = probabilities["test"]["llm"]

    candidate_cal_metrics["xgboost"] = metrics_from_probs(cal_xgb, cal_labels, n_classes)
    candidate_test_probs["xgboost"] = probabilities["test"]["xgb"]

    minimum_alpha = 0.0
    maximum_alpha = 1.0 - cfg.fusion_min_semantic_weight
    alphas = np.linspace(minimum_alpha, maximum_alpha, cfg.fusion_grid_points)

    best_alpha = None
    best_alpha_metrics = None
    best_alpha_score = -np.inf

    for alpha in alphas:
        fused = alpha * cal_xgb + (1.0 - alpha) * cal_llm
        fused = fused / fused.sum(axis=1, keepdims=True)
        metrics = metrics_from_probs(fused, cal_labels, n_classes)
        score = selection_score(metrics, cfg)
        if score > best_alpha_score:
            best_alpha_score = score
            best_alpha = float(alpha)
            best_alpha_metrics = metrics

    candidate_name_fusion = f"fusion_alpha_{best_alpha:.3f}"
    candidate_cal_metrics[candidate_name_fusion] = best_alpha_metrics

    test_alpha_probs = (
        best_alpha * probabilities["test"]["xgb"]
        + (1.0 - best_alpha) * probabilities["test"]["llm"]
    )
    test_alpha_probs /= test_alpha_probs.sum(axis=1, keepdims=True)
    candidate_test_probs[candidate_name_fusion] = test_alpha_probs

    eps = 1e-8
    stack_X_cal = np.concatenate(
        [np.log(cal_xgb + eps), np.log(cal_llm + eps)],
        axis=1,
    )
    stacker = LogisticRegression(
        C=10.0,
        max_iter=1000,
        solver="lbfgs",
        multi_class="multinomial",
        n_jobs=-1,
        random_state=seed,
    )
    stacker.fit(stack_X_cal, cal_labels)
    stack_cal_probs = align_probabilities(
        stacker.predict_proba(stack_X_cal),
        stacker.classes_,
        n_classes,
    )
    candidate_cal_metrics["stacking"] = metrics_from_probs(stack_cal_probs, cal_labels, n_classes)

    stack_X_test = np.concatenate(
        [
            np.log(probabilities["test"]["xgb"] + eps),
            np.log(probabilities["test"]["llm"] + eps),
        ],
        axis=1,
    )
    candidate_test_probs["stacking"] = align_probabilities(
        stacker.predict_proba(stack_X_test),
        stacker.classes_,
        n_classes,
    )

    cal_comparison = pd.DataFrame(candidate_cal_metrics).T.sort_values(
        cfg.fusion_selection_metric,
        ascending=False,
    )
    cal_comparison.to_csv(run_dir / "candidate_calibration_comparison.csv")

    selected_candidate = cal_comparison.index[0]
    final_probs = candidate_test_probs[selected_candidate]
    test_labels = probabilities["test"]["labels"]
    final_predictions = final_probs.argmax(axis=1)
    final_metrics = metrics_from_probs(final_probs, test_labels, n_classes)
    final_report = classification_report(
        test_labels,
        final_predictions,
        labels=np.arange(n_classes),
        target_names=class_names,
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    final_cm = confusion_matrix(test_labels, final_predictions, labels=np.arange(n_classes))

    test_results = {
        name: metrics_from_probs(probs, test_labels, n_classes)
        for name, probs in candidate_test_probs.items()
    }
    test_comparison = pd.DataFrame(test_results).T.sort_values(
        cfg.fusion_selection_metric,
        ascending=False,
    )
    test_comparison.to_csv(run_dir / "candidate_test_comparison.csv")

    branch_summary = pd.DataFrame({
        "MiniLM-LoRA": test_results["minilm"],
        "XGBoost": test_results["xgboost"],
        "Final": test_results[selected_candidate],
    }).T
    branch_summary["delta_weighted_f1_vs_minilm"] = (
        branch_summary["weighted_f1"] - branch_summary.loc["MiniLM-LoRA", "weighted_f1"]
    )
    branch_summary["delta_macro_f1_vs_minilm"] = (
        branch_summary["macro_f1"] - branch_summary.loc["MiniLM-LoRA", "macro_f1"]
    )
    if "macro_f1_all" in branch_summary.columns:
        branch_summary["delta_macro_f1_all_vs_minilm"] = (
            branch_summary["macro_f1_all"] - branch_summary.loc["MiniLM-LoRA", "macro_f1_all"]
        )
    branch_summary.to_csv(run_dir / "branch_summary.csv")

    # Sauvegardes des modèles et prédictions.
    adapter_dir = run_dir / "minilm_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    llm_model.save_pretrained(adapter_dir)
    llm_tokenizer.save_pretrained(adapter_dir)
    adapter_size_mb = dir_size_mb(adapter_dir)

    joblib.dump(xgb_model, run_dir / "xgboost_model.joblib")
    joblib.dump(stacker, run_dir / "fusion_stacker.joblib")
    joblib.dump(artifacts["label_encoder"], run_dir / "label_encoder.joblib")
    joblib.dump(artifacts["imputer"], run_dir / "imputer.joblib")
    joblib.dump(artifacts["variance_filter"], run_dir / "variance_filter.joblib")
    joblib.dump(artifacts["scaler"], run_dir / "scaler.joblib")
    joblib.dump(artifacts["all_feature_columns"], run_dir / "input_features.joblib")
    joblib.dump(numeric_features, run_dir / "numeric_features.joblib")
    joblib.dump(text_features, run_dir / "text_features.joblib")

    prediction_df = pd.DataFrame({
        "true_label": [class_names[index] for index in test_labels],
        "predicted_label": [class_names[index] for index in final_predictions],
        "confidence": final_probs.max(axis=1),
    })
    for class_index, class_name in enumerate(class_names):
        prediction_df[f"p_{class_name}"] = final_probs[:, class_index]
    prediction_df.to_csv(run_dir / "test_predictions.csv", index=False)

    with open(run_dir / "classification_report.json", "w", encoding="utf-8") as handle:
        json.dump(final_report, handle, indent=2, ensure_ascii=False)
    np.save(run_dir / "confusion_matrix.npy", final_cm)

    metadata_dict = {
        "protocol": protocol,
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": best_alpha,
        "calibration_selection_size": int(len(cal_select_idx)),
        "conformal_calibration_size": int(len(cal_conformal_idx)),
        "text_features": text_features,
        "numeric_features": numeric_features,
        "token_audit": token_audit,
        "param_stats": llm_bundle["param_stats"],
        "llm_train_seconds": llm_bundle["train_seconds"],
        "xgb_train_seconds": xgb_train_seconds,
        "peak_vram_gib": llm_bundle["peak_vram_gib"],
        "adapter_size_mb": adapter_size_mb,
    }
    with open(run_dir / "run_metadata.json", "w", encoding="utf-8") as handle:
        json.dump(metadata_dict, handle, indent=2, ensure_ascii=False)

    # Libération mémoire.
    del llm_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    summary_row = {
        "protocol": protocol,
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": best_alpha,
        "n_classes": n_classes,
        "classes": " | ".join(class_names),
        "numeric_features": " | ".join(numeric_features),
        "text_features": " | ".join(text_features),
        **llm_bundle["param_stats"],
        "llm_train_seconds": llm_bundle["train_seconds"],
        "xgb_train_seconds": xgb_train_seconds,
        "peak_vram_gib": llm_bundle["peak_vram_gib"],
        "adapter_size_mb": adapter_size_mb,
        **token_audit,
        **{f"final_{key}": value for key, value in final_metrics.items()},
        **{f"minilm_test_{key}": value for key, value in test_results["minilm"].items()},
        **{f"xgboost_test_{key}": value for key, value in test_results["xgboost"].items()},
    }

    print("\nRÉSULTAT FINAL")
    print(pd.Series(summary_row).head(30))
    print("\nRapport final :")
    print(classification_report(
        test_labels,
        final_predictions,
        labels=np.arange(n_classes),
        target_names=class_names,
        digits=4,
        zero_division=0,
    ))

    return {
        "summary_row": summary_row,
        "split_distribution_df": split_distribution_df,
        "importance_df": importance_df.assign(protocol=protocol, seed=seed),
        "token_audit": token_audit,
        "history_df": llm_bundle["history_df"].assign(protocol=protocol, seed=seed),
        "branch_summary": branch_summary.assign(protocol=protocol, seed=seed),
        "cal_comparison": cal_comparison.assign(protocol=protocol, seed=seed),
        "test_comparison": test_comparison.assign(protocol=protocol, seed=seed),
        "confusion_matrix": final_cm,
        "class_names": class_names,
        "run_dir": str(run_dir),
        "selected_candidate": selected_candidate,
    }

## 6 — Fonctions de calibration conforme et d’explicabilité v6

In [ ]:

# ================================================================
# TRUSTWORTHINESS v6 : calibration, conformité et explicabilité
# ================================================================

def transform_splits_from_saved_preprocessing(
    splits: Dict[str, pd.DataFrame],
    run_dir: Path,
) -> Dict[str, object]:
    label_encoder = joblib.load(run_dir / "label_encoder.joblib")
    imputer = joblib.load(run_dir / "imputer.joblib")
    variance_filter = joblib.load(run_dir / "variance_filter.joblib")
    scaler = joblib.load(run_dir / "scaler.joblib")
    input_features = joblib.load(run_dir / "input_features.joblib")
    numeric_features = joblib.load(run_dir / "numeric_features.joblib")
    text_features = joblib.load(run_dir / "text_features.joblib")

    variance_features = [
        feature for feature, keep in zip(input_features, variance_filter.get_support()) if keep
    ]
    variance_index = {name: idx for idx, name in enumerate(variance_features)}
    numeric_positions = [variance_index[name] for name in numeric_features]
    text_positions = [numeric_features.index(name) for name in text_features]

    arrays = {}
    for split_name, split_df in splits.items():
        X = split_df[input_features].to_numpy(dtype=np.float64)
        X_imp = imputer.transform(X)
        X_var = variance_filter.transform(X_imp)
        X_num = X_var[:, numeric_positions].astype(np.float32)
        X_scaled = scaler.transform(X_num).astype(np.float32)
        arrays[split_name] = {
            "raw_numeric": X_num,
            "scaled_numeric": X_scaled,
            "scaled_text": X_scaled[:, text_positions],
            "y": label_encoder.transform(split_df[LABEL_COL]).astype(np.int64),
        }

    return {
        "arrays": arrays,
        "label_encoder": label_encoder,
        "class_names": label_encoder.classes_.tolist(),
        "n_classes": len(label_encoder.classes_),
        "numeric_features": numeric_features,
        "text_features": text_features,
    }


def load_minilm_adapter(run_dir: Path, n_classes: int, class_names: List[str]):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_checkpoint,
        num_labels=n_classes,
        id2label={i: name for i, name in enumerate(class_names)},
        label2id={name: i for i, name in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )
    model = PeftModel.from_pretrained(base_model, run_dir / "minilm_adapter").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(run_dir / "minilm_adapter")
    model.eval()
    return model, tokenizer


def get_candidate_probabilities(
    candidate: str,
    llm_probs: np.ndarray,
    xgb_probs: np.ndarray,
    stacker,
    best_alpha: float,
    n_classes: int,
) -> np.ndarray:
    if candidate == "minilm":
        probs = llm_probs
    elif candidate == "xgboost":
        probs = xgb_probs
    elif candidate.startswith("fusion_alpha_"):
        probs = best_alpha * xgb_probs + (1.0 - best_alpha) * llm_probs
    elif candidate == "stacking":
        eps = 1e-8
        features = np.concatenate([np.log(xgb_probs + eps), np.log(llm_probs + eps)], axis=1)
        probs = align_probabilities(stacker.predict_proba(features), stacker.classes_, n_classes)
    else:
        raise ValueError(f"Candidat inconnu : {candidate}")
    probs = np.asarray(probs, dtype=np.float64)
    return probs / probs.sum(axis=1, keepdims=True)


def fit_temperature(probabilities: np.ndarray, labels: np.ndarray) -> float:
    logits = torch.tensor(np.log(np.clip(probabilities, 1e-12, 1.0)), dtype=torch.float64)
    targets = torch.tensor(labels, dtype=torch.long)
    log_temperature = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=80, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        temperature = torch.exp(log_temperature).clamp(0.05, 20.0)
        loss = F.cross_entropy(logits / temperature, targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.exp(log_temperature.detach()).clamp(0.05, 20.0).item())


def apply_temperature(probabilities: np.ndarray, temperature: float) -> np.ndarray:
    logits = np.log(np.clip(probabilities, 1e-12, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exp_values = np.exp(logits)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def conformal_true_scores(
    probabilities: np.ndarray,
    labels: np.ndarray,
    method: str,
    raps_lambda: float,
    k_reg: int,
) -> np.ndarray:
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(len(order))[:, None], order] = np.arange(probabilities.shape[1])[None, :]
    true_rank = ranks[np.arange(len(labels)), labels]
    scores = cumulative[np.arange(len(labels)), true_rank]
    if method == "RAPS":
        scores = scores + raps_lambda * np.maximum(true_rank + 1 - k_reg, 0)
    return scores


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    level = min(1.0, math.ceil((n + 1) * (1.0 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def build_prediction_sets(
    probabilities: np.ndarray,
    qhat: float,
    method: str,
    raps_lambda: float,
    k_reg: int,
) -> List[List[int]]:
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    rank_numbers = np.arange(1, probabilities.shape[1] + 1)
    scores = cumulative.copy()
    if method == "RAPS":
        scores = scores + raps_lambda * np.maximum(rank_numbers - k_reg, 0)[None, :]

    sets = []
    for i in range(len(probabilities)):
        include = scores[i] <= qhat
        include[0] = True
        sets.append(order[i][include].tolist())
    return sets


def evaluate_prediction_sets(
    prediction_sets: List[List[int]],
    labels: np.ndarray,
    class_names: List[str],
    abstain_size: int,
) -> Tuple[Dict[str, float], pd.DataFrame]:
    sizes = np.asarray([len(values) for values in prediction_sets], dtype=np.int64)
    covered = np.asarray([int(label in values) for label, values in zip(labels, prediction_sets)])
    overall = {
        "coverage": float(covered.mean()),
        "average_set_size": float(sizes.mean()),
        "median_set_size": float(np.median(sizes)),
        "singleton_rate": float(np.mean(sizes == 1)),
        "ambiguity_rate": float(np.mean((sizes > 1) & (sizes <= abstain_size))),
        "abstention_rate": float(np.mean(sizes > abstain_size)),
    }
    rows = []
    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        rows.append({
            "class": class_name,
            "support": int(mask.sum()),
            "coverage": float(covered[mask].mean()) if mask.any() else np.nan,
            "average_set_size": float(sizes[mask].mean()) if mask.any() else np.nan,
        })
    return overall, pd.DataFrame(rows)


def stratified_explanation_indices(labels: np.ndarray, n_samples: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    present_classes = np.unique(labels)
    per_class = max(1, n_samples // max(1, len(present_classes)))
    selected = []
    for class_index in present_classes:
        indices = np.flatnonzero(labels == class_index)
        take = min(per_class, len(indices))
        selected.extend(rng.choice(indices, size=take, replace=False).tolist())
    remaining = n_samples - len(selected)
    if remaining > 0:
        pool = np.setdiff1d(np.arange(len(labels)), np.asarray(selected, dtype=np.int64))
        if len(pool):
            selected.extend(rng.choice(pool, size=min(remaining, len(pool)), replace=False).tolist())
    return np.asarray(sorted(selected[:n_samples]), dtype=np.int64)


def get_tree_shap_values(xgb_model, X: np.ndarray, n_classes: int) -> np.ndarray:
    raw = xgb_model.get_booster().predict(
        xgb.DMatrix(X),
        pred_contribs=True,
        strict_shape=True,
    )
    raw = np.asarray(raw)
    if raw.ndim == 3:
        return raw[:, :, :-1]
    if raw.ndim == 2 and n_classes > 1:
        return raw.reshape(len(X), n_classes, -1)[:, :, :-1]
    raise RuntimeError(f"Forme TreeSHAP inattendue : {raw.shape}")


def minilm_predict_values(model, tokenizer, text_features: List[str], values_batch: np.ndarray) -> np.ndarray:
    prompts = [compact_prompt(text_features, values) for values in values_batch]
    encoded = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=CFG.max_length,
        return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()


def deterministic_template_explanation(
    predicted_label: str,
    confidence: float,
    conformal_labels: List[str],
    shap_evidence: List[Tuple[str, float]],
    occlusion_evidence: List[Tuple[str, float]],
) -> str:
    numeric_text = ", ".join(f"[{name}] ({value:+.3f})" for name, value in shap_evidence)
    semantic_text = ", ".join(f"[{name}] (drop={value:.3f})" for name, value in occlusion_evidence)
    uncertainty = (
        f"The conformal set contains only {predicted_label}."
        if len(conformal_labels) == 1
        else f"The decision is uncertain among: {', '.join(conformal_labels)}."
    )
    return (
        f"The predicted class is {predicted_label} with confidence {confidence:.3f}. "
        f"The strongest numeric evidence is {numeric_text}. "
        f"The MiniLM evidence is {semantic_text}. {uncertainty}"
    )


_EXPLAINER_CACHE = {}

def get_explanation_slm():
    checkpoint = CFG.explanation_model_checkpoint
    if checkpoint not in _EXPLAINER_CACHE:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(DEVICE)
        model.eval()
        _EXPLAINER_CACHE[checkpoint] = (model, tokenizer)
    return _EXPLAINER_CACHE[checkpoint]


def generate_slm_explanations(prompts: List[str]) -> Tuple[List[str], float]:
    model, tokenizer = get_explanation_slm()
    start = time.perf_counter()
    outputs = []
    batch_size = 8
    for start_index in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start_index:start_index + batch_size]
        encoded = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=CFG.explanation_max_new_tokens,
                do_sample=False,
                num_beams=2,
                repetition_penalty=1.1,
            )
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    seconds = time.perf_counter() - start
    return outputs, seconds


def citation_metrics(text: str, allowed_features: List[str]) -> Dict[str, float]:
    citations = re.findall(r"\[([a-zA-Z0-9_]+)\]", text)
    allowed = set(allowed_features)
    valid = [citation for citation in citations if citation in allowed]
    unsupported = [citation for citation in citations if citation not in allowed]
    return {
        "citation_count": len(citations),
        "valid_citation_rate": len(valid) / max(1, len(citations)),
        "unsupported_citation_rate": len(unsupported) / max(1, len(citations)),
        "evidence_coverage": len(set(valid)) / max(1, len(allowed)),
    }


## 7 — Exécution complète des expériences

In [ ]:
all_summaries = []
all_histories = []
all_importances = []
all_branch_summaries = []
all_split_distributions = []
all_cal_comparisons = []
all_test_comparisons = []
all_token_audits = []
confusion_registry = {}

for protocol in CFG.protocols:
    for seed in CFG.seeds:
        result = run_single_experiment(
            data=df,
            protocol=protocol,
            seed=seed,
            cfg=CFG,
            output_dir=MODEL_DIR,
        )
        all_summaries.append(result["summary_row"])
        all_histories.append(result["history_df"])
        all_importances.append(result["importance_df"])
        all_branch_summaries.append(result["branch_summary"])
        all_split_distributions.append(result["split_distribution_df"])
        all_cal_comparisons.append(result["cal_comparison"])
        all_test_comparisons.append(result["test_comparison"])
        all_token_audits.append(result["token_audit"])
        confusion_registry[(protocol, seed)] = {
            "cm": result["confusion_matrix"],
            "class_names": result["class_names"],
            "selected_candidate": result["selected_candidate"],
            "run_dir": result["run_dir"],
        }

summaries_df = pd.DataFrame(all_summaries)
histories_df = pd.concat(all_histories, ignore_index=True)
feature_importances_df = pd.concat(all_importances, ignore_index=True)
branch_summaries_df = pd.concat(all_branch_summaries, ignore_index=False)
split_distributions_df = pd.concat(all_split_distributions, ignore_index=True)
cal_comparisons_df = pd.concat(all_cal_comparisons, ignore_index=False)
test_comparisons_df = pd.concat(all_test_comparisons, ignore_index=False)
token_audits_df = pd.DataFrame(all_token_audits)

summaries_df.to_csv(TABLE_DIR / "all_run_summaries.csv", index=False)
histories_df.to_csv(TABLE_DIR / "all_training_histories.csv", index=False)
feature_importances_df.to_csv(TABLE_DIR / "all_feature_importances.csv", index=False)
branch_summaries_df.to_csv(TABLE_DIR / "all_branch_summaries.csv")
split_distributions_df.to_csv(TABLE_DIR / "all_split_distributions.csv", index=False)
cal_comparisons_df.to_csv(TABLE_DIR / "all_calibration_comparisons.csv")
test_comparisons_df.to_csv(TABLE_DIR / "all_test_comparisons.csv")
token_audits_df.to_csv(TABLE_DIR / "all_token_audits.csv", index=False)

display(summaries_df.head())
print("Expériences terminées :", len(summaries_df))

## 8 — Tableaux agrégés (moyenne ± écart-type)

In [ ]:

summary_metrics = [
    "final_accuracy",
    "final_macro_f1_present",
    "final_macro_f1_all",
    "final_weighted_f1",
    "final_balanced_accuracy",
    "final_mcc",
    "final_label_space_coverage",
    "llm_train_seconds",
    "xgb_train_seconds",
    "peak_vram_gib",
    "adapter_size_mb",
    "truncation_rate",
]

aggregate_protocol = (
    summaries_df
    .groupby("protocol")[summary_metrics]
    .agg(["mean", "std"])
)

branch_metrics = [
    "accuracy",
    "macro_f1",
    "macro_f1_all",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
    "label_space_coverage",
]
aggregate_branch = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
    .groupby(["protocol", "branch"])[branch_metrics]
    .agg(["mean", "std"])
)

llm_efficiency_table = summaries_df[[
    "protocol",
    "seed",
    "llm_total_parameters",
    "llm_trainable_parameters",
    "llm_trainable_percent",
    "adapter_size_mb",
    "llm_train_seconds",
    "peak_vram_gib",
    "mean_tokens",
    "p95_tokens",
    "truncation_rate",
]].copy()

aggregate_protocol.to_csv(TABLE_DIR / "aggregate_by_protocol.csv")
aggregate_branch.to_csv(TABLE_DIR / "aggregate_by_branch_and_protocol.csv")
llm_efficiency_table.to_csv(TABLE_DIR / "llm_efficiency_per_run.csv", index=False)

display(aggregate_protocol)
display(aggregate_branch)
display(llm_efficiency_table.head())


## 9 — Graphes : performances finales par protocole

In [ ]:
plot_df = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
)

metrics_to_plot = [
    ("weighted_f1", "Weighted-F1"),
    ("macro_f1", "Macro-F1"),
    ("mcc", "MCC"),
    ("balanced_accuracy", "Balanced Accuracy"),
]

for metric_key, metric_label in metrics_to_plot:
    grouped = (
        plot_df.groupby(["protocol", "branch"])[metric_key]
        .agg(["mean", "std"])
        .reset_index()
    )

    protocols = grouped["protocol"].unique().tolist()
    branches = grouped["branch"].unique().tolist()
    x = np.arange(len(protocols))
    width = 0.8 / max(1, len(branches))

    plt.figure(figsize=(12, 5))
    for i, branch in enumerate(branches):
        subset = grouped[grouped["branch"] == branch].set_index("protocol").reindex(protocols)
        plt.bar(
            x + (i - (len(branches)-1)/2) * width,
            subset["mean"].to_numpy(),
            width=width,
            yerr=subset["std"].fillna(0.0).to_numpy(),
            capsize=4,
            label=branch,
        )

    plt.xticks(x, protocols, rotation=0)
    plt.ylabel(metric_label)
    plt.ylim(0.0, 1.02)
    plt.title(f"{metric_label} — moyenne ± écart-type")
    plt.legend()
    save_figure(f"performance_{metric_key}_by_protocol")
    plt.show()

## 10 — Graphe : comparaison frugalité / performance

In [ ]:
final_plot_df = summaries_df.copy()
final_plot_df["final_weighted_f1_pct"] = 100 * final_plot_df["final_weighted_f1"]
final_plot_df["llm_train_minutes"] = final_plot_df["llm_train_seconds"] / 60.0

plt.figure(figsize=(10, 6))
for protocol in final_plot_df["protocol"].unique():
    subset = final_plot_df[final_plot_df["protocol"] == protocol]
    plt.scatter(
        subset["llm_train_minutes"],
        subset["final_weighted_f1_pct"],
        s=80,
        label=protocol,
    )
    for _, row in subset.iterrows():
        plt.annotate(
            f"seed={int(row['seed'])}",
            (row["llm_train_minutes"], row["final_weighted_f1_pct"]),
            fontsize=9,
            xytext=(3, 3),
            textcoords="offset points",
        )

plt.xlabel("Temps d'entraînement MiniLM (minutes)")
plt.ylabel("Weighted-F1 final (%)")
plt.title("Frugalité vs performance finale")
plt.legend()
save_figure("efficiency_vs_weighted_f1")
plt.show()

## 11 — Graphe : courbes d’apprentissage MiniLM

In [ ]:
for protocol in histories_df["protocol"].unique():
    subset_protocol = histories_df[histories_df["protocol"] == protocol]

    plt.figure(figsize=(12, 5))
    for seed in sorted(subset_protocol["seed"].unique()):
        subset = subset_protocol[subset_protocol["seed"] == seed]
        plt.plot(subset["epoch"], subset["train_loss"], marker="o", label=f"train_loss seed={seed}")
    plt.xlabel("Époque")
    plt.ylabel("Loss")
    plt.title(f"MiniLM — courbes de loss ({protocol})")
    plt.legend(ncol=2, fontsize=9)
    save_figure(f"training_loss_{protocol}")
    plt.show()

    plt.figure(figsize=(12, 5))
    for seed in sorted(subset_protocol["seed"].unique()):
        subset = subset_protocol[subset_protocol["seed"] == seed]
        plt.plot(subset["epoch"], subset["val_bal_macro_f1"], marker="o", label=f"macro_f1_bal seed={seed}")
    plt.xlabel("Époque")
    plt.ylabel("Macro-F1")
    plt.ylim(0.0, 1.02)
    plt.title(f"MiniLM — Macro-F1 de validation équilibrée ({protocol})")
    plt.legend(ncol=2, fontsize=9)
    save_figure(f"training_macro_f1_bal_{protocol}")
    plt.show()

## 12 — Graphes : audit des tokens et distribution des splits

In [ ]:
for protocol in token_audits_df["protocol"].unique():
    representative_seed = token_audits_df[token_audits_df["protocol"] == protocol].sort_values("seed").iloc[0]["seed"]
    run_dir = MODEL_DIR / f"{protocol}__seed_{int(representative_seed)}"
    audit_path = run_dir / "token_audit.csv"
    audit_df = pd.read_csv(audit_path)

    plt.figure(figsize=(7, 4))
    plt.bar(
        ["mean", "median", "p95", "max"],
        [
            float(audit_df["mean_tokens"].iloc[0]),
            float(audit_df["median_tokens"].iloc[0]),
            float(audit_df["p95_tokens"].iloc[0]),
            float(audit_df["max_tokens"].iloc[0]),
        ],
    )
    plt.axhline(float(audit_df["max_length"].iloc[0]), linestyle="--")
    plt.ylabel("Nombre de tokens")
    plt.title(f"Audit de longueur des prompts — {protocol}")
    save_figure(f"token_length_audit_{protocol}")
    plt.show()

dist_plot = (
    split_distributions_df
    .groupby(["protocol", "split", "label"], as_index=False)["count"]
    .mean()
)

for protocol in dist_plot["protocol"].unique():
    subset = dist_plot[dist_plot["protocol"] == protocol]
    pivot = subset.pivot(index="label", columns="split", values="count").fillna(0.0)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    plt.figure(figsize=(12, 6))
    x = np.arange(len(pivot.index))
    splits_present = pivot.columns.tolist()
    width = 0.8 / len(splits_present)
    for i, split_name in enumerate(splits_present):
        plt.bar(
            x + (i - (len(splits_present)-1)/2) * width,
            pivot[split_name].to_numpy(),
            width=width,
            label=split_name,
        )
    plt.xticks(x, pivot.index, rotation=45, ha="right")
    plt.ylabel("Nombre moyen de flux")
    plt.title(f"Distribution moyenne des splits — {protocol}")
    plt.legend()
    save_figure(f"split_distribution_{protocol}")
    plt.show()

## 13 — Graphes : matrices de confusion des meilleurs runs

In [ ]:
for protocol in CFG.protocols:
    protocol_runs = summaries_df[summaries_df["protocol"] == protocol].copy()
    best_idx = protocol_runs["final_weighted_f1"].idxmax()
    best_row = protocol_runs.loc[best_idx]
    key = (protocol, int(best_row["seed"]))
    cm_info = confusion_registry[key]
    cm = cm_info["cm"]
    class_names = cm_info["class_names"]

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=False,
        fmt="d",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
    )
    plt.xlabel("Prédit")
    plt.ylabel("Vrai")
    plt.title(
        f"Matrice de confusion — {protocol} | seed={int(best_row['seed'])} | "
        f"{cm_info['selected_candidate']}"
    )
    save_figure(f"confusion_matrix_best_{protocol}")
    plt.show()

## 14 — Graphes : importance des caractéristiques XGBoost

In [ ]:
importance_mean = (
    feature_importances_df
    .groupby(["protocol", "feature"], as_index=False)["importance"]
    .mean()
)

for protocol in importance_mean["protocol"].unique():
    subset = (
        importance_mean[importance_mean["protocol"] == protocol]
        .sort_values("importance", ascending=False)
        .head(20)
    )

    plt.figure(figsize=(9, 7))
    plt.barh(subset["feature"][::-1], subset["importance"][::-1])
    plt.xlabel("Importance moyenne")
    plt.title(f"Top-20 importances XGBoost — {protocol}")
    save_figure(f"xgb_feature_importance_{protocol}")
    plt.show()

## 15 — Résumé final et fichiers utiles

In [ ]:
best_overall_idx = summaries_df["final_weighted_f1"].idxmax()
best_overall = summaries_df.loc[best_overall_idx]

protocol_summary = (
    summaries_df.groupby("protocol")[["final_weighted_f1", "final_macro_f1", "final_mcc", "llm_train_seconds", "xgb_train_seconds"]]
    .agg(["mean", "std"])
)

print("Meilleur run global :")
display(best_overall.to_frame(name="value"))

print("\nRésumé par protocole :")
display(protocol_summary)

summary_export = {
    "best_overall_protocol": best_overall["protocol"],
    "best_overall_seed": int(best_overall["seed"]),
    "best_overall_selected_candidate": best_overall["selected_candidate"],
    "best_overall_weighted_f1": float(best_overall["final_weighted_f1"]),
    "best_overall_macro_f1": float(best_overall["final_macro_f1"]),
    "best_overall_mcc": float(best_overall["final_mcc"]),
}

with open(TABLE_DIR / "best_overall_summary.json", "w", encoding="utf-8") as handle:
    json.dump(summary_export, handle, indent=2, ensure_ascii=False)

print("\nFichiers principaux :")
print("  Résumés CSV   :", TABLE_DIR / "all_run_summaries.csv")
print("  Figures       :", FIG_DIR)
print("  Modèles       :", MODEL_DIR)
print("  Résumé JSON   :", TABLE_DIR / "best_overall_summary.json")

## 16 — Analyse conforme et explicabilité par SLM

In [ ]:

trust_summary_rows = []
trust_class_coverage_rows = []
xai_detail_rows = []
verbalizer_rows = []

for protocol in CFG.protocols:
    protocol_runs = summaries_df[summaries_df["protocol"] == protocol].copy()
    if protocol_runs.empty:
        continue
    best_idx = protocol_runs["final_weighted_f1"].idxmax()
    best_row = protocol_runs.loc[best_idx]
    seed = int(best_row["seed"])
    run_dir = MODEL_DIR / f"{protocol}__seed_{seed}"

    print("\n" + "=" * 90)
    print(f"TRUSTWORTHINESS : {protocol} | seed={seed}")
    print("=" * 90)

    splits = make_splits(
        data=df,
        protocol=protocol,
        seed=seed,
        label_col=LABEL_COL,
        train_fraction=CFG.train_fraction,
        val_fraction=CFG.val_fraction,
        cal_fraction=CFG.cal_fraction,
        test_fraction=CFG.test_fraction,
    )
    context = transform_splits_from_saved_preprocessing(splits, run_dir)
    arrays = context["arrays"]
    class_names = context["class_names"]
    n_classes = context["n_classes"]
    numeric_features = context["numeric_features"]
    text_features = context["text_features"]

    with open(run_dir / "run_metadata.json", "r", encoding="utf-8") as handle:
        metadata_dict = json.load(handle)
    selected_candidate = metadata_dict["selected_candidate"]
    best_alpha = float(metadata_dict["best_alpha_numeric"])

    xgb_model = joblib.load(run_dir / "xgboost_model.joblib")
    stacker = joblib.load(run_dir / "fusion_stacker.joblib")
    minilm_model, minilm_tokenizer = load_minilm_adapter(run_dir, n_classes, class_names)

    probabilities = {}
    for split_name in ["cal", "test"]:
        loader = make_full_llm_loader(
            arrays,
            split_name,
            minilm_tokenizer,
            text_features,
            CFG,
        )
        llm_probs, labels, _ = collect_llm_probs(minilm_model, loader, DEVICE)
        xgb_probs = np.asarray(xgb_model.predict_proba(arrays[split_name]["raw_numeric"]), dtype=np.float64)
        xgb_probs /= xgb_probs.sum(axis=1, keepdims=True)
        final_probs = get_candidate_probabilities(
            selected_candidate,
            llm_probs,
            xgb_probs,
            stacker,
            best_alpha,
            n_classes,
        )
        probabilities[split_name] = {
            "llm": llm_probs,
            "xgb": xgb_probs,
            "final": final_probs,
            "labels": labels,
        }

    partition = np.load(run_dir / "calibration_partition_indices.npz")
    selection_idx = partition["selection_indices"]
    conformal_idx = partition["conformal_indices"]

    temperature = fit_temperature(
        probabilities["cal"]["final"][selection_idx],
        probabilities["cal"]["labels"][selection_idx],
    )
    cal_probs_temp = apply_temperature(probabilities["cal"]["final"][conformal_idx], temperature)
    test_probs_temp = apply_temperature(probabilities["test"]["final"], temperature)
    cal_labels_conf = probabilities["cal"]["labels"][conformal_idx]
    test_labels = probabilities["test"]["labels"]

    method_sets = {}
    for method in ["APS", "RAPS"]:
        scores = conformal_true_scores(
            cal_probs_temp,
            cal_labels_conf,
            method=method,
            raps_lambda=CFG.raps_lambda,
            k_reg=CFG.raps_k_reg,
        )
        qhat = conformal_quantile(scores, CFG.conformal_alpha)
        prediction_sets = build_prediction_sets(
            test_probs_temp,
            qhat=qhat,
            method=method,
            raps_lambda=CFG.raps_lambda,
            k_reg=CFG.raps_k_reg,
        )
        overall, classwise = evaluate_prediction_sets(
            prediction_sets,
            test_labels,
            class_names,
            CFG.conformal_abstain_size,
        )
        method_sets[method] = prediction_sets
        trust_summary_rows.append({
            "protocol": protocol,
            "seed": seed,
            "selected_candidate": selected_candidate,
            "temperature": temperature,
            "method": method,
            "qhat": qhat,
            **overall,
        })
        classwise["protocol"] = protocol
        classwise["seed"] = seed
        classwise["method"] = method
        trust_class_coverage_rows.append(classwise)

    # RAPS est utilisé pour les explications finales.
    raps_sets = method_sets["RAPS"]
    explanation_indices = stratified_explanation_indices(
        test_labels,
        CFG.explanation_samples,
        seed + 71,
    )

    X_explain = arrays["test"]["raw_numeric"][explanation_indices]
    shap_values = get_tree_shap_values(xgb_model, X_explain, n_classes)
    final_predictions = test_probs_temp.argmax(axis=1)
    train_medians = np.median(arrays["train"]["raw_numeric"], axis=0)

    slm_prompts = []
    slm_row_indices = []
    templates = []

    for local_position, test_index in enumerate(tqdm(explanation_indices, desc=f"XAI {protocol}")):
        predicted_class = int(final_predictions[test_index])
        predicted_label = class_names[predicted_class]
        confidence = float(test_probs_temp[test_index, predicted_class])

        # TreeSHAP
        class_shap = shap_values[local_position, predicted_class]
        shap_order = np.argsort(-np.abs(class_shap))[:CFG.explanation_top_k]
        shap_evidence = [(numeric_features[j], float(class_shap[j])) for j in shap_order]

        modified_numeric = arrays["test"]["raw_numeric"][test_index].copy()
        modified_numeric[shap_order] = train_medians[shap_order]
        numeric_after = xgb_model.predict_proba(modified_numeric[None, :])[0]
        xgb_original_class_prob = float(probabilities["test"]["xgb"][test_index, predicted_class])
        shap_fidelity_drop = xgb_original_class_prob - float(numeric_after[predicted_class])

        # Occlusion MiniLM
        original_text_values = arrays["test"]["scaled_text"][test_index].copy()
        variants = np.repeat(original_text_values[None, :], len(text_features), axis=0)
        variants[np.arange(len(text_features)), np.arange(len(text_features))] = 0.0
        start_occ = time.perf_counter()
        variant_probs = minilm_predict_values(minilm_model, minilm_tokenizer, text_features, variants)
        occlusion_seconds = time.perf_counter() - start_occ
        llm_original_prob = float(probabilities["test"]["llm"][test_index, predicted_class])
        drops = llm_original_prob - variant_probs[:, predicted_class]
        occ_order = np.argsort(-drops)[:CFG.explanation_top_k]
        occlusion_evidence = [(text_features[j], float(drops[j])) for j in occ_order]
        combined_values = original_text_values.copy()
        combined_values[occ_order] = 0.0
        combined_prob = minilm_predict_values(
            minilm_model,
            minilm_tokenizer,
            text_features,
            combined_values[None, :],
        )[0, predicted_class]
        occlusion_fidelity_drop = llm_original_prob - float(combined_prob)

        shap_names = [name for name, _ in shap_evidence]
        occ_names = [name for name, _ in occlusion_evidence]
        agreement = len(set(shap_names) & set(occ_names)) / max(1, len(set(shap_names) | set(occ_names)))

        conformal_labels = [class_names[index] for index in raps_sets[test_index]]
        template = deterministic_template_explanation(
            predicted_label,
            confidence,
            conformal_labels,
            shap_evidence,
            occlusion_evidence,
        )
        templates.append(template)

        allowed_features = list(dict.fromkeys(shap_names + occ_names))
        prompt = (
            f"Write exactly three short {CFG.explanation_language} sentences for a cybersecurity analyst. "
            "Use only the supplied evidence. Cite feature names exactly in square brackets. "
            "Do not invent causes, protocols, ports, or features. "
            f"Prediction: {predicted_label}. Confidence: {confidence:.3f}. "
            f"Conformal alternatives: {', '.join(conformal_labels)}. "
            f"Numeric evidence: {', '.join(f'[{n}] contribution {v:+.3f}' for n, v in shap_evidence)}. "
            f"MiniLM occlusion evidence: {', '.join(f'[{n}] drop {v:.3f}' for n, v in occlusion_evidence)}. "
            "If the conformal set has more than one class, explicitly state that the decision is uncertain."
        )
        slm_prompts.append(prompt)
        slm_row_indices.append(len(xai_detail_rows))

        xai_detail_rows.append({
            "protocol": protocol,
            "seed": seed,
            "test_index": int(test_index),
            "true_label": class_names[int(test_labels[test_index])],
            "predicted_label": predicted_label,
            "confidence": confidence,
            "conformal_set": " | ".join(conformal_labels),
            "tree_shap_features": " | ".join(shap_names),
            "tree_shap_values": " | ".join(f"{value:.6f}" for _, value in shap_evidence),
            "tree_shap_fidelity_drop": shap_fidelity_drop,
            "occlusion_features": " | ".join(occ_names),
            "occlusion_drops": " | ".join(f"{value:.6f}" for _, value in occlusion_evidence),
            "occlusion_fidelity_drop": occlusion_fidelity_drop,
            "occlusion_latency_seconds": occlusion_seconds,
            "cross_method_jaccard": agreement,
            "template_explanation": template,
            "allowed_features": " | ".join(allowed_features),
        })

    slm_count = min(CFG.slm_explanation_samples, len(slm_prompts))
    generated, generation_seconds = generate_slm_explanations(slm_prompts[:slm_count])
    per_slm_seconds = generation_seconds / max(1, slm_count)

    protocol_xai_start = len(xai_detail_rows) - len(explanation_indices)
    for offset, row_index in enumerate(range(protocol_xai_start, protocol_xai_start + len(explanation_indices))):
        row = xai_detail_rows[row_index]
        allowed = row["allowed_features"].split(" | ") if row["allowed_features"] else []
        template_metrics = citation_metrics(row["template_explanation"], allowed)
        verbalizer_rows.append({
            "protocol": protocol,
            "seed": seed,
            "test_index": row["test_index"],
            "method": "Deterministic template",
            "text": row["template_explanation"],
            "latency_seconds": 0.0,
            "word_count": len(row["template_explanation"].split()),
            "predicted_label_present": float(row["predicted_label"].lower() in row["template_explanation"].lower()),
            **template_metrics,
        })
        if offset < slm_count:
            slm_text = generated[offset]
            slm_metrics = citation_metrics(slm_text, allowed)
            verbalizer_rows.append({
                "protocol": protocol,
                "seed": seed,
                "test_index": row["test_index"],
                "method": "FLAN-T5-small",
                "text": slm_text,
                "latency_seconds": per_slm_seconds,
                "word_count": len(slm_text.split()),
                "predicted_label_present": float(row["predicted_label"].lower() in slm_text.lower()),
                **slm_metrics,
            })

    del minilm_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

trust_summary_df = pd.DataFrame(trust_summary_rows)
trust_class_coverage_df = pd.concat(trust_class_coverage_rows, ignore_index=True)
xai_details_df = pd.DataFrame(xai_detail_rows)
verbalizer_comparison_df = pd.DataFrame(verbalizer_rows)

trust_summary_df.to_csv(TABLE_DIR / "conformal_summary.csv", index=False)
trust_class_coverage_df.to_csv(TABLE_DIR / "conformal_classwise_coverage.csv", index=False)
xai_details_df.to_csv(TABLE_DIR / "explainability_details.csv", index=False)
verbalizer_comparison_df.to_csv(TABLE_DIR / "slm_verbalizer_comparison.csv", index=False)

xai_method_summary_df = pd.DataFrame([
    {
        "method": "TreeSHAP",
        "mean_fidelity_drop": xai_details_df["tree_shap_fidelity_drop"].mean(),
        "mean_latency_seconds": np.nan,
        "mean_cross_method_jaccard": xai_details_df["cross_method_jaccard"].mean(),
    },
    {
        "method": "MiniLM feature occlusion",
        "mean_fidelity_drop": xai_details_df["occlusion_fidelity_drop"].mean(),
        "mean_latency_seconds": xai_details_df["occlusion_latency_seconds"].mean(),
        "mean_cross_method_jaccard": xai_details_df["cross_method_jaccard"].mean(),
    },
])
xai_method_summary_df.to_csv(TABLE_DIR / "xai_method_comparison.csv", index=False)

display(trust_summary_df)
display(xai_method_summary_df)
display(verbalizer_comparison_df.groupby("method").agg({
    "valid_citation_rate": "mean",
    "unsupported_citation_rate": "mean",
    "evidence_coverage": "mean",
    "latency_seconds": "mean",
    "word_count": "mean",
    "predicted_label_present": "mean",
}))


## 17 — Graphiques de conformité et d’explicabilité

In [ ]:

# 1. Couverture vs taille moyenne
for protocol in trust_summary_df["protocol"].unique():
    subset = trust_summary_df[trust_summary_df["protocol"] == protocol]
    plt.figure(figsize=(8, 5))
    for _, row in subset.iterrows():
        plt.scatter(row["average_set_size"], row["coverage"], s=110)
        plt.annotate(row["method"], (row["average_set_size"], row["coverage"]), xytext=(5, 5), textcoords="offset points")
    plt.axhline(1.0 - CFG.conformal_alpha, linestyle="--", label="Couverture cible")
    plt.xlabel("Taille moyenne de l'ensemble")
    plt.ylabel("Couverture")
    plt.ylim(0.0, 1.02)
    plt.title(f"Conformité : couverture vs efficacité — {protocol}")
    plt.legend()
    save_figure(f"conformal_coverage_vs_size_{protocol}")
    plt.show()

# 2. Couverture par classe RAPS
for protocol in trust_class_coverage_df["protocol"].unique():
    subset = trust_class_coverage_df[
        (trust_class_coverage_df["protocol"] == protocol)
        & (trust_class_coverage_df["method"] == "RAPS")
        & (trust_class_coverage_df["support"] > 0)
    ]
    plt.figure(figsize=(12, 5))
    plt.bar(subset["class"], subset["coverage"])
    plt.axhline(1.0 - CFG.conformal_alpha, linestyle="--")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Couverture RAPS")
    plt.ylim(0.0, 1.02)
    plt.title(f"Couverture conforme par classe — {protocol}")
    save_figure(f"raps_classwise_coverage_{protocol}")
    plt.show()

# 3. Fidélité des méthodes XAI
plt.figure(figsize=(8, 5))
plt.bar(xai_method_summary_df["method"], xai_method_summary_df["mean_fidelity_drop"])
plt.ylabel("Baisse moyenne de probabilité après retrait Top-k")
plt.title("Fidélité comparative des méthodes d'explication")
plt.xticks(rotation=20, ha="right")
save_figure("xai_method_faithfulness")
plt.show()

# 4. Ancrage des verbaliseurs
verbalizer_agg = verbalizer_comparison_df.groupby("method", as_index=False).agg({
    "valid_citation_rate": "mean",
    "unsupported_citation_rate": "mean",
    "evidence_coverage": "mean",
    "predicted_label_present": "mean",
})
plt.figure(figsize=(9, 5))
x = np.arange(len(verbalizer_agg))
width = 0.25
plt.bar(x - width, verbalizer_agg["valid_citation_rate"], width=width, label="Citations valides")
plt.bar(x, verbalizer_agg["evidence_coverage"], width=width, label="Couverture des preuves")
plt.bar(x + width, verbalizer_agg["predicted_label_present"], width=width, label="Classe mentionnée")
plt.xticks(x, verbalizer_agg["method"])
plt.ylim(0.0, 1.05)
plt.ylabel("Score moyen")
plt.title("Ancrage des explications textuelles")
plt.legend()
save_figure("slm_explanation_groundedness")
plt.show()

# 5. Latence des verbaliseurs
verbalizer_latency = verbalizer_comparison_df.groupby("method", as_index=False)["latency_seconds"].mean()
plt.figure(figsize=(8, 5))
plt.bar(verbalizer_latency["method"], verbalizer_latency["latency_seconds"])
plt.ylabel("Secondes par explication")
plt.title("Latence de génération des explications")
save_figure("slm_explanation_latency")
plt.show()


## 18 — Conseils d’utilisation pour l’article

### Mesures à rapporter
- **Weighted-F1**
- **Macro-F1**
- **MCC**
- **Balanced Accuracy**
- temps d’entraînement
- VRAM
- taux de troncature

### Figures déjà prêtes
- `global_class_distribution`
- `performance_weighted_f1_by_protocol`
- `performance_macro_f1_by_protocol`
- `performance_mcc_by_protocol`
- `efficiency_vs_weighted_f1`
- `training_loss_<protocol>`
- `training_macro_f1_bal_<protocol>`
- `token_length_audit_<protocol>`
- `split_distribution_<protocol>`
- `confusion_matrix_best_<protocol>`
- `xgb_feature_importance_<protocol>`

### Lecture correcte du résultat
Si la fusion sélectionnée est `xgboost`, `fusion_alpha_1.000` ou `stacking` avec `alpha=1.0`, cela signifie que la branche numérique porte l’essentiel de la performance.
Dans ce cas, MiniLM doit être présenté comme **branche sémantique frugale complémentaire**, et non comme source unique du score final.

### Modules de confiance v6
- APS et RAPS sont comparés sur une moitié de `CAL` réservée exclusivement à la conformité.
- TreeSHAP explique la branche XGBoost.
- L’occlusion explique la branche MiniLM.
- FLAN-T5-small ne décide pas de la classe : il verbalise uniquement les preuves déjà calculées.
- Le taux de citations non supportées doit être rapporté comme mesure d’hallucination explicative.


## 19 — Tableaux corrigés, statistiques et visualisations circulaires

Cette section ajoute les éléments souvent attendus dans un papier fort : métriques corrigées pour support partiel des classes, tests statistiques simples, visualisations circulaires, radar et tableaux article-ready.

In [ ]:

article_protocol_table = (
    summaries_df
    .groupby("protocol")[[
        "final_accuracy",
        "final_weighted_f1",
        "final_macro_f1_present",
        "final_macro_f1_all",
        "final_balanced_accuracy",
        "final_mcc",
        "final_label_space_coverage",
        "llm_train_seconds",
        "xgb_train_seconds",
        "peak_vram_gib",
        "adapter_size_mb",
    ]]
    .agg(["mean", "std"])
)
article_protocol_table.to_csv(TABLE_DIR / "article_protocol_table_corrected.csv")
display(article_protocol_table)

article_branch_table = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
    .groupby(["protocol", "branch"])[[
        "accuracy",
        "weighted_f1",
        "macro_f1",
        "macro_f1_all",
        "balanced_accuracy",
        "mcc",
        "label_space_coverage",
    ]]
    .agg(["mean", "std"])
)
article_branch_table.to_csv(TABLE_DIR / "article_branch_table_corrected.csv")
display(article_branch_table)


In [ ]:

significance_rows = []
branch_long = branch_summaries_df.reset_index().rename(columns={"index": "branch"})

if wilcoxon is not None:
    for protocol in branch_long["protocol"].unique():
        subset = branch_long[branch_long["protocol"] == protocol].copy()
        for metric in ["weighted_f1", "macro_f1", "macro_f1_all", "mcc"]:
            pivot = subset.pivot(index="seed", columns="branch", values=metric)
            comparisons = [("Final", "XGBoost"), ("Final", "MiniLM-LoRA"), ("XGBoost", "MiniLM-LoRA")]
            for left, right in comparisons:
                if left not in pivot.columns or right not in pivot.columns:
                    continue
                paired = pivot[[left, right]].dropna()
                if len(paired) < 2:
                    stat, pvalue = np.nan, np.nan
                else:
                    try:
                        stat, pvalue = wilcoxon(paired[left], paired[right], alternative="two-sided")
                    except Exception:
                        stat, pvalue = np.nan, np.nan
                significance_rows.append({
                    "protocol": protocol,
                    "metric": metric,
                    "left": left,
                    "right": right,
                    "n_pairs": int(len(paired)),
                    "left_mean": float(paired[left].mean()) if len(paired) else np.nan,
                    "right_mean": float(paired[right].mean()) if len(paired) else np.nan,
                    "delta_mean": float((paired[left] - paired[right]).mean()) if len(paired) else np.nan,
                    "wilcoxon_stat": float(stat) if stat == stat else np.nan,
                    "wilcoxon_pvalue": float(pvalue) if pvalue == pvalue else np.nan,
                })
else:
    print("SciPy indisponible : tests statistiques sautés.")

significance_df = pd.DataFrame(significance_rows)
significance_df.to_csv(TABLE_DIR / "statistical_tests.csv", index=False)
display(significance_df)


In [ ]:

macro_compare = summaries_df.groupby("protocol")[["final_macro_f1_present", "final_macro_f1_all"]].agg(["mean", "std"])
protocols = macro_compare.index.tolist()
x = np.arange(len(protocols))
width = 0.34

present_mean = macro_compare[("final_macro_f1_present", "mean")].to_numpy()
present_std = macro_compare[("final_macro_f1_present", "std")].fillna(0.0).to_numpy()
all_mean = macro_compare[("final_macro_f1_all", "mean")].to_numpy()
all_std = macro_compare[("final_macro_f1_all", "std")].fillna(0.0).to_numpy()

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, present_mean, width=width, yerr=present_std, capsize=4, label="Macro-F1 present")
plt.bar(x + width/2, all_mean, width=width, yerr=all_std, capsize=4, label="Macro-F1 all")
plt.xticks(x, protocols)
plt.ylim(0.0, 1.02)
plt.ylabel("Score")
plt.title("Macro-F1 corrigé : classes présentes vs espace global des labels")
plt.legend()
save_figure("macrof1_present_vs_all")
plt.show()


In [ ]:

protocol_best = summaries_df.loc[summaries_df.groupby("protocol")["final_weighted_f1"].idxmax()].copy()
fig, axes = plt.subplots(1, len(protocol_best), figsize=(6 * len(protocol_best), 5))
if len(protocol_best) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, protocol_best.iterrows()):
    present = int(row["final_present_class_count"])
    absent = int(row["n_classes"] - row["final_present_class_count"])
    ax.pie(
        [present, absent],
        labels=[f"Présentes ({present})", f"Absentes ({absent})"],
        startangle=90,
        wedgeprops={"width": 0.42, "edgecolor": "white"},
        autopct="%1.0f%%",
    )
    ax.set_title(f"Couverture des classes\n{row['protocol']}")
plt.tight_layout()
save_figure("donut_label_space_coverage")
plt.show()


In [ ]:

row = summaries_df.loc[summaries_df["final_weighted_f1"].idxmax()]
trainable = float(row["llm_trainable_parameters"])
frozen = float(row["llm_total_parameters"] - row["llm_trainable_parameters"])
plt.figure(figsize=(6, 6))
plt.pie(
    [trainable, frozen],
    labels=[f"Entraînables\n{trainable:,.0f}", f"Gelés\n{frozen:,.0f}"],
    startangle=90,
    wedgeprops={"width": 0.42, "edgecolor": "white"},
    autopct="%1.2f%%",
)
plt.title("Frugalité LoRA du meilleur MiniLM")
save_figure("donut_lora_parameter_efficiency")
plt.show()


In [ ]:

def make_radar(ax, values_dict, categories, title):
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 1.0)
    for name, values in values_dict.items():
        vals = list(values) + [values[0]]
        ax.plot(angles, vals, linewidth=2, label=name)
        ax.fill(angles, vals, alpha=0.08)
    ax.set_title(title, y=1.10)

radar_source = (
    branch_summaries_df.reset_index().rename(columns={"index": "branch"})
    .groupby(["protocol", "branch"])[["weighted_f1", "macro_f1", "macro_f1_all", "balanced_accuracy", "mcc"]]
    .mean()
    .reset_index()
)
radar_metrics = ["weighted_f1", "macro_f1", "macro_f1_all", "balanced_accuracy", "mcc"]
radar_labels = ["Weighted-F1", "Macro-F1 present", "Macro-F1 all", "Balanced Acc", "MCC"]
fig, axes = plt.subplots(1, len(CFG.protocols), figsize=(7 * len(CFG.protocols), 6), subplot_kw={"projection": "polar"})
if len(CFG.protocols) == 1:
    axes = [axes]
for ax, protocol in zip(axes, CFG.protocols):
    subset = radar_source[radar_source["protocol"] == protocol]
    values_dict = {}
    for _, row in subset.iterrows():
        values_dict[row["branch"]] = [float(row[m]) for m in radar_metrics]
    make_radar(ax, values_dict, radar_labels, f"Radar multi-métrique — {protocol}")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.92])
save_figure("radar_branch_comparison_by_protocol")
plt.show()


In [ ]:

verbalizer_agg = verbalizer_comparison_df.groupby("method", as_index=False).agg({
    "valid_citation_rate": "mean",
    "unsupported_citation_rate": "mean",
    "evidence_coverage": "mean",
    "predicted_label_present": "mean",
    "latency_seconds": "mean",
})
verbalizer_agg["groundedness"] = 1.0 - verbalizer_agg["unsupported_citation_rate"]
max_latency = max(verbalizer_agg["latency_seconds"].max(), 1e-6)
verbalizer_agg["latency_score"] = 1.0 - (verbalizer_agg["latency_seconds"] / max_latency)

categories = ["Valid citations", "Groundedness", "Evidence coverage", "Predicted label", "Latency score"]
fig = plt.figure(figsize=(7, 6))
ax = plt.subplot(111, projection="polar")
values_dict = {}
for _, row in verbalizer_agg.iterrows():
    values_dict[row["method"]] = [
        float(row["valid_citation_rate"]),
        float(row["groundedness"]),
        float(row["evidence_coverage"]),
        float(row["predicted_label_present"]),
        float(row["latency_score"]),
    ]
make_radar(ax, values_dict, categories, "Radar de qualité des explications textuelles")
plt.legend(loc="upper right", bbox_to_anchor=(1.32, 1.10))
save_figure("radar_verbalizer_quality")
plt.show()


In [ ]:

trust_plot = trust_summary_df.copy()
fig, axes = plt.subplots(len(CFG.protocols), 2, figsize=(10, 5 * len(CFG.protocols)))
if len(CFG.protocols) == 1:
    axes = np.array([axes])
for i, protocol in enumerate(CFG.protocols):
    for j, method in enumerate(["APS", "RAPS"]):
        ax = axes[i, j]
        subset = trust_plot[(trust_plot["protocol"] == protocol) & (trust_plot["method"] == method)]
        if subset.empty:
            ax.axis("off")
            continue
        row = subset.iloc[0]
        sizes = [row["singleton_rate"], row["ambiguity_rate"], row["abstention_rate"]]
        ax.pie(
            sizes,
            labels=["Singleton", "Ambigu", "Abstention"],
            startangle=90,
            wedgeprops={"width": 0.42, "edgecolor": "white"},
            autopct="%1.1f%%",
        )
        ax.set_title(f"{protocol} — {method}\nCoverage={row['coverage']:.3f} | Avg size={row['average_set_size']:.2f}")
plt.tight_layout()
save_figure("donut_conformal_efficiency")
plt.show()


In [ ]:

best_efficiency = (
    summaries_df.sort_values(["protocol", "final_weighted_f1"], ascending=[True, False])
    .groupby("protocol")
    .head(1)
    .copy()
)
plt.figure(figsize=(9, 6))
plt.scatter(
    best_efficiency["llm_train_seconds"],
    best_efficiency["final_weighted_f1"],
    s=best_efficiency["peak_vram_gib"] * 900,
    c=np.arange(len(best_efficiency)),
    alpha=0.75,
)
for _, row in best_efficiency.iterrows():
    plt.annotate(
        f"{row['protocol']}\nVRAM={row['peak_vram_gib']:.2f} GiB\nAdapter={row['adapter_size_mb']:.2f} MB",
        (row["llm_train_seconds"], row["final_weighted_f1"]),
        xytext=(10, 10),
        textcoords="offset points",
    )
plt.xlabel("Temps d'entraînement MiniLM (s)")
plt.ylabel("Weighted-F1 final")
plt.ylim(0.0, 1.02)
plt.title("Efficacité du LLM : coût d'entraînement vs performance")
save_figure("llm_efficiency_scatter")
plt.show()


## 20 — Recommandations de rédaction pour le best paper

- **Rapporter à la fois** `Macro-F1 present` et `Macro-F1 all` pour les protocoles à support partiel.
- Mettre en avant l'écart **IID vs temporel** comme preuve de robustesse méthodologique.
- Présenter le système comme un **hybride frugal et fiable** : XGBoost pour la force numérique, MiniLM-LoRA pour la sémantique, APS/RAPS pour l'incertitude, TreeSHAP/Occlusion/FLAN-T5 pour l'explicabilité.
- Utiliser les figures générées dans `figures/` et les tableaux `article_*` dans `tables/` comme matériau principal de l'article.